# 08 - Testing

In [2]:

# 08: Testing
# Integrity checks on every pipeline output. Each check either passes or is listed
# as a failure; the final cell raises an error if anything failed, so this notebook
# can be re-run after any change to catch regressions.

import json, os
import numpy as np
import pandas as pd

results = []
def check(name, condition, detail=""):
    results.append({"Check": name, "Passed": bool(condition), "Detail": detail})

clean = pd.read_csv("data/all_months_clean.csv")
feat = pd.read_csv("data/all_months_features.csv")
fc = pd.read_csv("results/month_11_forecast.csv")
val = pd.read_csv("results/validation_month_9_predictions.csv")
comp = pd.read_csv("results/baseline_vs_rf_vs_lightgbm_validation.csv")
with open("results/lightgbm_best_params.json") as f:
    params = json.load(f)

In [3]:

# Data-stage checks.
check("Cleaned data has 798 rows", len(clean) == 798, f"rows={len(clean)}")
check("No zero or negative Avg_Price", (clean["Avg_Price"].dropna() > 0).all())
check("Zero Min/Max prices were converted to missing",
      (clean["Min_Price"].dropna() > 0).all() and (clean["Max_Price"].dropna() > 0).all())
check("Canonical product-name fixes applied", "Brocauli" not in set(clean["Product_Name"]))
check("Stray 'Naew' column removed", "Naew" not in clean.columns)
check("n_months_present between 1 and 10", clean["n_months_present"].between(1, 10).all())
check("Feature file keeps all cleaned rows", len(feat) == len(clean))
for col in ["Avg_Price_lag1", "price_spread", "sin_month", "cos_month", "import_ratio"]:
    check(f"Feature column present: {col}", col in feat.columns)

In [4]:

# Forecast checks: full coverage, valid ordering, no negative prices.
check("Forecast covers every product", set(fc["Product_Name"]) == set(clean["Product_Name"]),
      f"{fc['Product_Name'].nunique()} / {clean['Product_Name'].nunique()}")
check("One forecast row per product", fc["Product_Name"].is_unique)
for model in ["RF_Predicted", "LightGBM_Predicted", "Baseline"]:
    mn, av, mx = (fc[f"{model}_{k}_Price"] if model != "Baseline" else fc[f"Baseline_{k}_Price"]
                  for k in ["Min", "Avg", "Max"])
    ok = ((mn <= av + 1e-9) & (av <= mx + 1e-9)) | mn.isna() | av.isna() | mx.isna()
    check(f"{model}: Min <= Avg <= Max for every row", ok.all())
check("LightGBM forecast has no missing values",
      fc[[c for c in fc.columns if c.startswith("LightGBM_")]].notna().all().all())
check("No negative forecast prices",
      (fc[[c for c in fc.columns if "Price" in c and c.endswith("Price")]].dropna() >= 0).all().all())
check("Forecast month follows last observed month",
      (fc["Forecast_Month_Idx"] == fc["Last_Observed_Month_Idx"] + 1).all())

In [5]:

# Validation / model-output checks.
check("Validation file has rows", len(val) > 0, f"rows={len(val)}")
check("Comparison table has 3 targets", len(comp) == 3)
check("LightGBM beats baseline MAE on every target", (comp["LightGBM MAE"] < comp["Baseline MAE"]).all())
check("Saved params contain the expected hyperparameters",
      {"n_estimators", "learning_rate", "num_leaves", "max_depth", "random_state"} <= set(params))
check("Random seed is 42", params.get("random_state") == 42)
for f in ["category_wise_error.csv", "unit_wise_error.csv", "volatile_product_analysis.csv",
          "lightgbm_feature_importance.csv"]:
    check(f"Output exists: {f}", os.path.exists(f"results/{f}"))

In [6]:

# Summary. Raises if any check failed so a broken pipeline cannot pass silently.
report = pd.DataFrame(results)
display(report)
n_fail = int((~report["Passed"]).sum())
print(f"{len(report) - n_fail} / {len(report)} checks passed")
assert n_fail == 0, f"{n_fail} check(s) failed - see table above"

,Check,Passed,Detail
0,Cleaned data has 798 rows,True,rows=798
1,No zero or negative Avg_Price,True,
2,Zero Min/Max prices were converted to missing,True,
3,Canonical product-name fixes applied,True,
4,Stray 'Naew' column removed,True,
5,n_months_present between 1 and 10,True,
6,Feature file keeps all cleaned rows,True,
7,Feature column present: Avg_Price_lag1,True,
8,Feature column present: price_spread,True,
9,Feature column present: sin_month,True,


29 / 29 checks passed
